# День 3 на Kaggle: шум таксономии, latency, экспорт на Hugging Face

Что здесь можно сделать: шаги 10 (шум таксономии), 08 (latency) и 09 (экспорт на Hub).
MLflow (шаг 07) здесь не нужен: его интерфейс открывается в браузере, поэтому запускай шаг 07 локально.

Перед запуском (Settings справа):
- Accelerator: **GPU T4 x2** (нужен для GPU-замера в шаге 08; используется одна карта)
- Internet: **On**
- Add Input (приватные датасеты):
  1. **код**: папки `src/` (с `infer_utils.py`, `project_utils.py`, `data_utils.py`, `train_utils.py`, `__init__.py`) и `scripts/`, плюс файл `MODEL_CARD.md`;
  2. **split**: `split_ids.parquet` и `label_names.json`;
  3. **результаты**: файл `v2_val_preds.npy` (из локальной папки results/; нужен для шага 10);
  4. **веса**: папка `v2_weighted` (модель и токенизатор).
- Для публикации на Hub: Add-ons -> Secrets -> добавь секрет `HF_TOKEN` (токен с правом Write). Не вставляй токен в ячейки.

Замеры latency зависят от железа: в README указывай, что это Kaggle (GPU T4 и CPU Kaggle), и не сравнивай с локальными цифрами.

In [ ]:
import os, sys, glob, shutil

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HOME"] = "/kaggle/temp/hf"

def find_one(pattern, what):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"Не найден {what}. Проверь Add Input."
    return hits[0]

SRC = os.path.dirname(find_one("/kaggle/input/**/data_utils.py", "src/data_utils.py"))
ROOT_IN = os.path.dirname(SRC)
for need in ("src/infer_utils.py", "scripts/08_benchmark.py", "scripts/09_export_to_hub.py",
             "scripts/10_taxonomy_noise.py", "MODEL_CARD.md"):
    assert os.path.exists(f"{ROOT_IN}/{need}"), f"В датасете с кодом нет {need}: загрузи новую версию датасета."

PROJECT = "/kaggle/working/project"
shutil.rmtree(PROJECT, ignore_errors=True)
for d in ("artifacts", "results", "models"):
    os.makedirs(f"{PROJECT}/{d}")
shutil.copytree(SRC, f"{PROJECT}/src")
shutil.copytree(f"{ROOT_IN}/scripts", f"{PROJECT}/scripts")
shutil.copy(f"{ROOT_IN}/MODEL_CARD.md", f"{PROJECT}/MODEL_CARD.md")
shutil.copy(find_one("/kaggle/input/**/split_ids.parquet", "split_ids.parquet"), f"{PROJECT}/artifacts/")
shutil.copy(find_one("/kaggle/input/**/label_names.json", "label_names.json"), f"{PROJECT}/artifacts/")

preds = glob.glob("/kaggle/input/**/v2_val_preds.npy", recursive=True)
if preds:
    shutil.copy(preds[0], f"{PROJECT}/results/v2_val_preds.npy")
else:
    print("v2_val_preds.npy не найден: шаг 10 работать не будет")

cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)]
cands = [c for c in cands if os.path.exists(f"{c}/model.safetensors") or os.path.exists(f"{c}/pytorch_model.bin")]
V2_DIR = next((c for c in cands if "v2" in c.lower()), cands[0] if cands else None)
print("Веса:", V2_DIR)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("Рабочая папка:", os.getcwd())

In [ ]:
!nvidia-smi -L
!nproc

In [ ]:
RUN_NOISE = True     # шаг 10: шум таксономии (нужен v2_val_preds.npy)
RUN_BENCH = True     # шаг 08: latency и размер (нужны веса)
RUN_EXPORT = True    # шаг 09: сборка hub_export (проверка экспорта)
PUSH = False         # True = загрузить на Hugging Face (нужен секрет HF_TOKEN и REPO_ID)
PRIVATE = True       # приватный репозиторий (позже можно сделать публичным в настройках на сайте)
REPO_ID = "ИМЯ_ПОЛЬЗОВАТЕЛЯ/marketplace-rubert-tiny2-category"

## Шаг 10: шум таксономии

Конфликты меток у одинаковых текстов в train, главные путаницы V2 и примеры ошибок. Скачай потом `results/error_examples.csv` и прочитай примеры глазами.

In [ ]:
if RUN_NOISE:
    !python scripts/10_taxonomy_noise.py

## Шаг 08: размер и latency

Замеры на Kaggle (T4 и общий CPU): укажи это в README. CPU-замеры на общей машине шумнее локальных.

In [ ]:
if RUN_BENCH and V2_DIR:
    !python scripts/08_benchmark.py --model-dir "{V2_DIR}"
elif RUN_BENCH:
    print("Веса не найдены: подключи датасет с v2_weighted")

## Шаг 09: экспорт на Hugging Face

Сначала пробный прогон (`PUSH = False`): собирается папка и проверяется, что модель отдаёт названия классов. Папка собирается в `/kaggle/temp`, чтобы не раздувать Output. Перед `PUSH = True` заполни метки в `MODEL_CARD.md` (в датасете с кодом) и проверь лицензии датасета и модели.

In [ ]:
if RUN_EXPORT and V2_DIR:
    cmd = f'python scripts/09_export_to_hub.py --model-dir "{V2_DIR}" --out /kaggle/temp/hub_export'
    if PUSH:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        cmd += f" --push --repo-id {REPO_ID}" + (" --private" if PRIVATE else "")
    !{cmd}
elif RUN_EXPORT:
    print("Веса не найдены: подключи датасет с v2_weighted")

In [ ]:
!ls -la results